# Gurchon Hall/Tabriz Assembly — First print inspection

This notebook serves the purpose to check first_print implementation _(commits [`cc85d81`](https://github.com/gurchon-hall/tabriz-assembly/commit/cc85d815907ffe9d67e21f8047ec446afcdb92c2) and [`cb53ebc`](https://github.com/gurchon-hall/tabriz-assembly/commit/cb53ebc35ae264e6290666aaa5b9dcb06e047e0c))_.

In [ ]:
import sys
from pathlib import Path

# Ajouter la racine du projet au path
PROJECT_ROOT = Path("__file__").parent.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, selectinload

from app.config import settings
from app.models.vtes import CryptCard, LibraryCard

logger = settings.log.get_logger(__name__)
engine = create_engine(settings.db.synchrone_url, echo=settings.db.database_echo)

print("Imports OK")


Imports OK


## Querying data

In [ ]:
with Session(engine) as session:
    crypt_cards = session.execute(
        select(CryptCard)
        .options(selectinload(CryptCard.first_print))
        .order_by(CryptCard.id)
    ).scalars().all()

print(len(crypt_cards), "crypt cards loaded")


1785 crypt cards loaded


In [ ]:
with Session(engine) as session:
    library_cards = session.execute(
        select(LibraryCard)
        .options(selectinload(LibraryCard.first_print))
        .order_by(LibraryCard.id)
    ).scalars().all()

print(len(library_cards), "library cards loaded")


2364 library cards loaded


In [ ]:
# Tri applicatif : release_date ASC (NULL en fin), puis nom ASC
def _sort_key(release_date: str | None, name: str) -> tuple[int, str, str]:
    """Trie : cartes avec date d'abord (ordre chronologique), NULL en fin, puis par nom."""
    return (0 if release_date is not None else 1, release_date or "", name)

def _rd(card: CryptCard | LibraryCard) -> str | None:
    fp = card.first_print
    return str(fp.release_date) if fp is not None else None

crypt_sorted = sorted(crypt_cards, key=lambda c: _sort_key(_rd(c), c.name))
library_sorted = sorted(library_cards, key=lambda c: _sort_key(_rd(c), c.name))

total = len(crypt_cards) + len(library_cards)


## Raw display of computed data

In [ ]:
_COL_ID = 8
_COL_NAME = 50
_COL_TYPE = 5   # "CRYPT" | "LIB"
_COL_GRP = 5    # group (crypt uniquement)
_COL_ADV = 4    # adv   (crypt uniquement)
_COL_SET = 20
_COL_DATE = 10

_SEP_WIDTH = _COL_ID + _COL_NAME + _COL_TYPE + _COL_GRP + _COL_ADV + _COL_SET + _COL_DATE + 6


In [6]:
def _header() -> str:
    return (
        f"{'ID':<{_COL_ID}} "
        f"{'Name':<{_COL_NAME}} "
        f"{'Kind':<{_COL_TYPE}} "
        f"{'Grp':<{_COL_GRP}} "
        f"{'Adv':<{_COL_ADV}} "
        f"{'First Set':<{_COL_SET}} "
        f"{'Release Date':<{_COL_DATE}}"
    )

def _separator() -> str:
    return "-" * _SEP_WIDTH

def _row(
    card_id: int,
    name: str,
    kind: str,
    group: str,
    adv: str,
    set_abbrev: str | None,
    release_date: str | None,
) -> str:
    return (
        f"{card_id:<{_COL_ID}} "
        f"{name:<{_COL_NAME}} "
        f"{kind:<{_COL_TYPE}} "
        f"{group:<{_COL_GRP}} "
        f"{adv:<{_COL_ADV}} "
        f"{(set_abbrev or 'NULL'):<{_COL_SET}} "
        f"{(release_date or 'NULL'):<{_COL_DATE}}"
    )


In [ ]:
print(_header())
print(_separator())

null_count = 0

for card in crypt_sorted:
    fp = card.first_print
    set_abbrev = fp.abbrev if fp is not None else None
    release_date = str(fp.release_date) if fp is not None else None
    if fp is None:
        null_count += 1
    print(_row(card.id, card.name, "CRYPT", card.group, "Y" if card.adv else "N", set_abbrev, release_date))

for card in library_sorted:
    fp = card.first_print
    set_abbrev = fp.abbrev if fp is not None else None
    release_date = str(fp.release_date) if fp is not None else None
    if fp is None:
        null_count += 1
    print(_row(card.id, card.name, "LIB", "-", "-", set_abbrev, release_date))

print(_separator())
total = len(crypt_cards) + len(library_cards)
print(f"Total : {total} cartes  |  Crypt : {len(crypt_cards)}  |  Library : {len(library_cards)}  |  first_print NULL : {null_count}")


ID       Name                                               Kind  Grp   Adv  First Set            Release Date
------------------------------------------------------------------------------------------------------------
200016   Adrianne                                           CRYPT 1     N    Jyhad                1994-08-16
200022   Agrippina                                          CRYPT 1     N    Jyhad                1994-08-16
200045   Aleph                                              CRYPT 1     N    Jyhad                1994-08-16
200077   Anastasia Grey                                     CRYPT 1     N    Jyhad                1994-08-16
200083   Andreas, The Bard of Crete                         CRYPT 1     N    Jyhad                1994-08-16
200088   Angel                                              CRYPT 1     N    Jyhad                1994-08-16
200095   Angus the Unruled                                  CRYPT 1     N    Jyhad                1994-08-16
200106   Anneke  